# BanglaLLM — Data Cleaning, Tokenizer Training & Final Dataset

**Pipeline overview:**
```
bengali_datasets/
├── titulm_local/        ← .arrow files
└── local_mc4_bn_data/   ← .arrow files
```
**Output:**
```
BanglaLLM/
├── cleaned/             ← cleaned + deduped HF dataset
├── tokenizer/           ← SentencePiece .model + .vocab
└── final_dataset/       ← chunked train/ and eval/ splits
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies (run once)
# ─────────────────────────────────────────────────────────────
# !pip install -q datasets sentencepiece transformers pyarrow huggingface_hub

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — Paths  (re-run this if you restart the kernel)
# ─────────────────────────────────────────────────────────────
import os

DATASET_FOLDER   = "./bengali_datasets"      # contains titulm_local/ and local_mc4_bn_data/
CLEANED_OUTPUT   = "./BanglaLLM/cleaned"
TOKENIZER_OUTPUT = "./BanglaLLM/tokenizer"
FINAL_OUTPUT     = "./BanglaLLM/final_dataset"
CHUNK_FILE       = f"{FINAL_OUTPUT}/chunks.arrow"
MAX_LENGTH       = 2048

os.makedirs(CLEANED_OUTPUT,   exist_ok=True)
os.makedirs(TOKENIZER_OUTPUT, exist_ok=True)
os.makedirs(FINAL_OUTPUT,     exist_ok=True)

print("Paths ready:")
print(f"  Input    : {DATASET_FOLDER}")
print(f"  Cleaned  : {CLEANED_OUTPUT}")
print(f"  Tokenizer: {TOKENIZER_OUTPUT}")
print(f"  Output   : {FINAL_OUTPUT}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Find all .arrow files
# ─────────────────────────────────────────────────────────────
import glob

arrow_files = glob.glob(f"{DATASET_FOLDER}/**/*.arrow", recursive=True)
print(f"Found {len(arrow_files)} .arrow file(s):\n")
for f in arrow_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Load .arrow files
#
# FIX: Strips each table to only the 'text' column before
# concat — prevents ArrowInvalid schema mismatch between
# c4 (text+timestamp+url) and mc4 (document_id+text) files.
# ─────────────────────────────────────────────────────────────
from datasets import Dataset, load_from_disk, concatenate_datasets
import pyarrow as pa

def load_arrow_data(dataset_folder):
    """
    Loads all .arrow files from dataset_folder into a single
    HuggingFace Dataset with one column: 'text'.
    Handles both HF save_to_disk format and raw arrow files.
    Handles mismatched schemas by normalising to text-only.
    """

    # ── Method A: HuggingFace save_to_disk() ─────────────────
    hf_folders = []
    for root, dirs, files in os.walk(dataset_folder):
        if "dataset_info.json" in files:
            hf_folders.append(root)

    if hf_folders:
        print(f"Found {len(hf_folders)} HuggingFace dataset folder(s):")
        loaded = []
        for folder in hf_folders:
            print(f"  Loading: {folder}")
            try:
                ds = load_from_disk(folder)
                print(f"    -> {len(ds):,} rows | columns: {ds.column_names}")
                # Keep only text column
                for col in ["text", "content", "body", "sentence", "passage"]:
                    if col in ds.column_names:
                        keep = [c for c in ds.column_names if c != col]
                        ds = ds.remove_columns(keep).rename_column(col, "text") \
                             if col != "text" else ds.remove_columns(
                             [c for c in ds.column_names if c != "text"])
                        break
                loaded.append(ds)
            except Exception as e:
                print(f"    -> Failed: {e}")
        if loaded:
            return concatenate_datasets(loaded) if len(loaded) > 1 else loaded[0]

    # ── Method B: Raw .arrow files ────────────────────────────
    arrow_files = glob.glob(f"{dataset_folder}/**/*.arrow", recursive=True)
    if not arrow_files:
        raise FileNotFoundError(f"No .arrow files found in {dataset_folder}")

    print(f"Loading {len(arrow_files)} raw .arrow file(s) with PyArrow...")
    all_tables = []

    for filepath in arrow_files:
        print(f"  Reading: {os.path.basename(filepath)}")
        table = None
        for opener in [pa.ipc.open_stream, pa.ipc.open_file]:
            try:
                with opener(filepath) as r:
                    table = r.read_all()
                break
            except pa.ArrowInvalid:
                continue

        if table is None:
            print(f"    -> Could not read, skipping.")
            continue

        print(f"    -> {table.num_rows:,} rows | columns: {table.schema.names}")

        # ── KEY FIX: normalise to text-only before appending ──
        text_col = None
        for candidate in ["text", "content", "body", "sentence", "passage"]:
            if candidate in table.schema.names:
                text_col = candidate
                break
        if text_col is None:
            for field in table.schema:
                if pa.types.is_string(field.type) or pa.types.is_large_string(field.type):
                    text_col = field.name
                    break
        if text_col is None:
            print(f"    -> No text column found, skipping.")
            continue

        if text_col != "text":
            print(f"    -> Renaming '{text_col}' -> 'text'")

        # Strip to text column only — fixes schema mismatch
        text_array = table.column(text_col).cast(pa.string())
        all_tables.append(pa.table({"text": text_array}))

    if not all_tables:
        raise ValueError("Could not read any .arrow files.")

    combined = pa.concat_tables(all_tables)   # safe — all schemas identical now
    return Dataset(combined)


raw_dataset = load_arrow_data(DATASET_FOLDER)

print(f"\nDataset loaded!")
print(f"Total samples : {len(raw_dataset):,}")
print(f"Columns       : {raw_dataset.column_names}")
print(f"\nFirst 3 samples:")
print("-" * 60)
for i in range(min(3, len(raw_dataset))):
    print(f"[{i}] {raw_dataset[i]['text'][:200]}\n")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Bengali text cleaning function
# Removes: HTML, URLs, emails, non-Bengali chars, short texts
# Keeps  : Bengali Unicode U+0980–U+09FF, punctuation, digits
# ─────────────────────────────────────────────────────────────
import re
import unicodedata

def clean_bengali_text(text):
    """Cleans a single Bengali string. Returns None if unusable."""
    if not isinstance(text, str):
        return None
    text = text.strip()
    if not text:
        return None

    text = re.sub(r'<[^>]+>', ' ', text)           # strip HTML
    text = re.sub(r'https?://\S+', ' ', text)      # strip URLs
    text = re.sub(r'www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)      # strip emails
    text = unicodedata.normalize('NFC', text)       # normalise Unicode

    # Keep Bengali block + punctuation + digits + whitespace
    text = re.sub(
        r'[^\u0980-\u09FF'
        r'\u0020\u0009\u000A'
        r'\.,!?\;\:\"\(\)\-\u2013\u0964\u0965'
        r"'"
        r'0-9]+',
        ' ', text
    )

    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()

    if len(text) < 20:
        return None

    bengali_chars = len(re.findall(r'[\u0980-\u09FF]', text))
    total_chars   = len(text.replace(' ', ''))
    if total_chars > 0 and (bengali_chars / total_chars) < 0.5:
        return None

    return text


# Quick sanity test
tests = [
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "<p>বাংলাদেশ</p> একটি সুন্দর দেশ। http://example.com",
    "This is English only.",
    "   ",
    "বাংলা",
]
print("Cleaning test:")
print("-" * 50)
for s in tests:
    print(f"In : {s[:70]}")
    print(f"Out: {clean_bengali_text(s)}\n")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Apply cleaning to full dataset
# num_proc=1 to avoid multiprocess issues on some systems
# ─────────────────────────────────────────────────────────────
import datasets
datasets.disable_caching()

def clean_batch(examples):
    return {"text": [clean_bengali_text(t) for t in examples["text"]]}

print("Cleaning dataset...")
cleaned_dataset = raw_dataset.map(
    clean_batch,
    batched=True,
    batch_size=500,
    num_proc=1,
    desc="Cleaning"
)

before = len(cleaned_dataset)
cleaned_dataset = cleaned_dataset.filter(
    lambda x: x["text"] is not None and len(x["text"]) >= 20,
    desc="Filtering short/empty"
)
after = len(cleaned_dataset)

print(f"\nCleaning complete!")
print(f"Before  : {before:,}")
print(f"After   : {after:,}")
print(f"Removed : {before - after:,}  ({(before - after) / before * 100:.1f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — Deduplicate using 80-char fingerprint
# ─────────────────────────────────────────────────────────────
print("Removing duplicates...")

seen   = set()
keep   = []

for i, example in enumerate(cleaned_dataset):
    fp = example["text"][:80].strip()
    if fp not in seen:
        seen.add(fp)
        keep.append(i)
    if (i + 1) % 100_000 == 0:
        print(f"  Checked {i+1:,} | Kept {len(keep):,}")

dedup_dataset = cleaned_dataset.select(keep)

print(f"\nDeduplication complete!")
print(f"Before  : {len(cleaned_dataset):,}")
print(f"After   : {len(dedup_dataset):,}")
print(f"Removed : {len(cleaned_dataset) - len(dedup_dataset):,} duplicates")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — Save cleaned dataset + 100-sample preview
# ─────────────────────────────────────────────────────────────
dedup_dataset.save_to_disk(CLEANED_OUTPUT)

sample_path = f"{CLEANED_OUTPUT}/sample_100.txt"
with open(sample_path, "w", encoding="utf-8") as f:
    for i in range(min(100, len(dedup_dataset))):
        f.write(dedup_dataset[i]["text"] + "\n\n" + "-"*60 + "\n\n")

print(f"Cleaned dataset -> {CLEANED_OUTPUT}")
print(f"Sample preview  -> {sample_path}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8b — Preview random samples after cleaning
# ─────────────────────────────────────────────────────────────
import random

NUM_SAMPLES = 20
total       = len(dedup_dataset)
indices     = random.sample(range(total), min(NUM_SAMPLES, total))

print(f"Showing {len(indices)} random samples from {total:,} cleaned rows")
print("=" * 70)

for rank, idx in enumerate(indices, 1):
    text = dedup_dataset[idx]["text"]
    print(f"\n[{rank:02d}] index={idx:,} | {len(text)} chars")
    print("-" * 70)
    print(text[:400])
    if len(text) > 400:
        print(f"... [{len(text)-400} more chars]")

# Length distribution
lengths = [len(dedup_dataset[i]["text"]) for i in range(min(5000, total))]
lengths.sort()
n = len(lengths)

print(f"\n{'=' * 70}")
print(f"Text length stats (over {n:,} samples):")
print(f"  Min    : {lengths[0]:,} chars")
print(f"  Max    : {lengths[-1]:,} chars")
print(f"  Mean   : {sum(lengths) // n:,} chars")
print(f"  Median : {lengths[n//2]:,} chars")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9 — Train SentencePiece Unigram tokenizer
#
# Why SentencePiece instead of LLaMA tokenizer:
#   LLaMA tokenizer was trained on English — it splits Bengali
#   characters into 3-4 tokens each. SentencePiece trained on
#   your corpus gives 2-3x better token efficiency for Bengali.
# ─────────────────────────────────────────────────────────────
import sentencepiece as spm
from datasets import load_from_disk

SP_MODEL_PREFIX = f"{TOKENIZER_OUTPUT}/bangla_spm"
VOCAB_SIZE      = 32000   # ← increase from 32k — full corpus warrants larger vocab
SAMPLE_SIZE     = 500_000    # rows to train SP on (None = full dataset)
SP_TRAIN_TXT    = "./BanglaLLM/sp_train_input.txt"

# ── Step 1: Write training text ───────────────────────────────
print("Reloading cleaned dataset...")
dedup_dataset = load_from_disk(CLEANED_OUTPUT)

sp_data = dedup_dataset.select(range(min(SAMPLE_SIZE, len(dedup_dataset)))) \
          if SAMPLE_SIZE else dedup_dataset

print(f"Writing {len(sp_data):,} lines for SentencePiece training...")
with open(SP_TRAIN_TXT, "w", encoding="utf-8") as f:
    for example in sp_data:
        line = example["text"].replace("\n", " ").strip()
        if line:
            f.write(line + "\n")
print(f"Training file written -> {SP_TRAIN_TXT}")

# ── Step 2: Train tokenizer ───────────────────────────────────
print(f"\nTraining SentencePiece Unigram (vocab={VOCAB_SIZE:,})...")
spm.SentencePieceTrainer.train(
    input                  = SP_TRAIN_TXT,
    model_prefix           = SP_MODEL_PREFIX,
    vocab_size             = VOCAB_SIZE,
    model_type             = "unigram",        # best for Bengali conjuncts
    character_coverage     = 0.9999,           # near-complete Bengali coverage
    pad_id                 = 0,
    unk_id                 = 1,
    bos_id                 = 2,
    eos_id                 = 3,
    pad_piece              = "<pad>",
    unk_piece              = "<unk>",
    bos_piece              = "<s>",
    eos_piece              = "</s>",
    byte_fallback          = True,             # zero OOV — encodes unseen chars as UTF-8 bytes
    split_digits           = True,             # tokenise digits individually
    num_threads            = os.cpu_count(),
    input_sentence_size   = SAMPLE_SIZE or 1_000_000,
    shuffle_input_sentence = True,
)

print(f"\nSentencePiece model  -> {SP_MODEL_PREFIX}.model")
print(f"Vocab file           -> {SP_MODEL_PREFIX}.vocab")

# ── Step 3: Quick test ────────────────────────────────────────
sp = spm.SentencePieceProcessor(model_file=f"{SP_MODEL_PREFIX}.model")

test_texts = [
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "ক্ষমা করবেন, আপনি কি বাংলায় কথা বলেন?",   # conjuncts
    "তাপমাত্রা ছিল ৩৭.৫ ডিগ্রি সেলসিয়াস।",      # digits
]

print("\nTokenizer test:")
print("-" * 60)
for text in test_texts:
    tokens   = sp.encode(text, out_type=str)
    ids      = sp.encode(text)
    decoded  = sp.decode(ids)
    lossless = decoded.strip() == text.strip()
    print(f"Text     : {text}")
    print(f"Tokens   : {tokens}")
    print(f"Count    : {len(tokens)} tokens | {len(text)/len(tokens):.1f} chars/token")
    print(f"Lossless : {'✓' if lossless else '✗ MISMATCH'}\n")

print(f"Vocab size : {sp.get_piece_size():,}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 10 — Reload cleaned dataset + tokenize
#
# FIX: tokenize_function loads sp from file path internally
# so it works correctly with num_proc=1 (SentencePiece is
# not safe to pickle across processes — path string is).
# ─────────────────────────────────────────────────────────────
import gc
import sentencepiece as spm
from datasets import load_from_disk

SP_MODEL_PATH = f"{TOKENIZER_OUTPUT}/bangla_spm.model"

# Reload sp in this cell explicitly
sp     = spm.SentencePieceProcessor(model_file=SP_MODEL_PATH)
BOS_ID = sp.bos_id()
EOS_ID = sp.eos_id()
print(f"SP model loaded | vocab size: {sp.get_piece_size():,}")

# Reload full cleaned dataset (no test slice)
print("Reloading cleaned dataset...")
dedup_dataset = load_from_disk(CLEANED_OUTPUT)
print(f"Total samples : {len(dedup_dataset):,}")

# Tokenize function — loads sp from path inside function
# so it survives any process boundary
def tokenize_function(examples):
    _sp = spm.SentencePieceProcessor(model_file=SP_MODEL_PATH)
    all_ids = []
    for text in examples["text"]:
        ids = [_sp.bos_id()] + _sp.encode(text) + [_sp.eos_id()]
        all_ids.append(ids)
    return {"input_ids": all_ids}

print("\nTokenizing...")

cols_to_drop = [c for c in dedup_dataset.column_names if c != "text"]
dedup_dataset = dedup_dataset.remove_columns(cols_to_drop)

tokenized_dataset = dedup_dataset.map(
    tokenize_function,
    batched   = True,
    batch_size = 500,
    remove_columns = ["text"],
    num_proc  = 1,           # MUST be 1 — SentencePiece not fork-safe
    desc      = "Tokenizing",
)

del dedup_dataset
gc.collect()

print(f"\nTokenization complete!")
print(f"Samples  : {len(tokenized_dataset):,}")
print(f"Columns  : {tokenized_dataset.column_names}")
sample_len = len(tokenized_dataset[0]["input_ids"])
print(f"Sample 0 : {sample_len} tokens")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 11 — Memory-safe streaming chunker
#
# Writes chunks directly to disk batch-by-batch using
# IPC file format (ipc.new_file). Never loads full data.
#
# FIX: Load-back uses ipc.open_file (not Dataset.from_file)
# because Dataset.from_file expects IPC stream format, but
# ipc.new_file writes IPC file format — different wire format.
# ─────────────────────────────────────────────────────────────
import pyarrow as pa
import pyarrow.ipc as ipc
import gc
from datasets import Dataset

# Remove any corrupt previous file
if os.path.exists(CHUNK_FILE):
    os.remove(CHUNK_FILE)
    print(f"Removed existing chunks.arrow")

BATCH_SIZE = 2000

schema = pa.schema([
    ("input_ids", pa.list_(pa.int32())),
    ("labels",    pa.list_(pa.int32())),
])

total_chunks = 0
leftover     = []

print(f"Streaming chunking -> {CHUNK_FILE}")
print(f"Input samples      : {len(tokenized_dataset):,}")
print(f"Chunk length       : {MAX_LENGTH} tokens")

with ipc.new_file(CHUNK_FILE, schema) as writer:
    for start in range(0, len(tokenized_dataset), BATCH_SIZE):
        end   = min(start + BATCH_SIZE, len(tokenized_dataset))
        batch = tokenized_dataset[start:end]

        for seq in batch["input_ids"]:
            leftover.extend(seq)

        n_chunks = len(leftover) // MAX_LENGTH

        if n_chunks > 0:
            usable   = leftover[:n_chunks * MAX_LENGTH]
            leftover = leftover[n_chunks * MAX_LENGTH:]
            rows     = [usable[i:i+MAX_LENGTH] for i in range(0, len(usable), MAX_LENGTH)]

            writer.write_batch(pa.record_batch({
                "input_ids": pa.array(rows, type=pa.list_(pa.int32())),
                "labels":    pa.array(rows, type=pa.list_(pa.int32())),
            }, schema=schema))

            total_chunks += n_chunks
            del rows, usable

        del batch
        gc.collect()
        print(f"  {end:,}/{len(tokenized_dataset):,} | chunks: {total_chunks:,}", end="\r")

print(f"\n\nChunking done!")
print(f"Chunks written         : {total_chunks:,}")
print(f"Total tokens           : {total_chunks * MAX_LENGTH:,}")
print(f"Leftover tokens skipped: {len(leftover):,}")

# ── Load back (FIX: use ipc.open_file not Dataset.from_file) ──
print("\nLoading chunks back (memory-mapped)...")
source          = pa.memory_map(CHUNK_FILE, "r")
table           = ipc.open_file(source).read_all()
chunked_dataset = Dataset(table)
del table
gc.collect()

print(f"Dataset ready : {len(chunked_dataset):,} chunks")
print(f"Sample check  : input_ids[0] length = {len(chunked_dataset[0]['input_ids'])}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 12 — Train / eval split  (95% / 5%)
# ─────────────────────────────────────────────────────────────
split         = chunked_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Split complete!")
print(f"Train : {len(train_dataset):,} chunks")
print(f"Eval  : {len(eval_dataset):,} chunks")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 13 — Save train / eval splits to disk
# ─────────────────────────────────────────────────────────────
from datasets import load_from_disk

train_dataset.save_to_disk(f"{FINAL_OUTPUT}/train")
eval_dataset.save_to_disk(f"{FINAL_OUTPUT}/eval")

print(f"Saved!")
print(f"  Train -> {FINAL_OUTPUT}/train")
print(f"  Eval  -> {FINAL_OUTPUT}/eval")

# Sanity check
check = load_from_disk(f"{FINAL_OUTPUT}/train")
print(f"\nSanity check : {len(check):,} train samples | columns: {check.column_names}")

print(f"\n{'=' * 60}")
print(f"Pipeline complete! Summary:")
print(f"  Cleaned samples : {len(load_from_disk(CLEANED_OUTPUT)):,}")
print(f"  Train chunks    : {len(train_dataset):,}")
print(f"  Eval chunks     : {len(eval_dataset):,}")
print(f"  Total tokens    : {(len(train_dataset) + len(eval_dataset)) * MAX_LENGTH:,}")
print(f"  Tokenizer vocab : {sp.get_piece_size():,}")
print(f"{'=' * 60}")